In [ ]:
import mltrainer
assert mltrainer.__version__ >= "0.2.5"

from pathlib import Path
from torch import nn
import torch

from src import datasets
from src.data_utils import (
    FullDataStreamer,
    split_train_validation,
    calculate_class_weights,
 )

In [ ]:
# Open config file
import tomllib

datadir = Path('../data')
configfile = Path("config.toml")

with configfile.open('rb') as f:
    config = tomllib.load(f)
print(config)

In [ ]:
# Dataset maken en trainingdata stratified splitsen in train/validation
full_traindataset = datasets.HeartDataset1D(trainfile, target="target")
testdataset = datasets.HeartDataset1D(testfile, target="target")
traindataset, validdataset = split_train_validation(
    full_traindataset, validation_fraction=0.2, seed=42
)
len(traindataset), len(validdataset), len(testdataset)


In [ ]:
# Data streamers. De officiële testset wordt niet als validation gebruikt.
trainstreamer = FullDataStreamer(traindataset, batch_size=32)
validstreamer = FullDataStreamer(validdataset, batch_size=32, shuffle=False)
teststreamer = FullDataStreamer(testdataset, batch_size=32, shuffle=False)
len(trainstreamer), len(validstreamer), len(teststreamer)

In [ ]:
trainloader = trainstreamer.stream()

x, y = next(iter(trainloader))

print(x.shape)
print(y.shape)

In [ ]:
# ============================================
# Experiment 1 - Baseline 1D CNN 
# ============================================

In [ ]:
from src.helpers import run_experiment
from src.networks import CNN1D


run_experiment(
    model_function=lambda: CNN1D(
        num_classes=5,
        filters=16,
        input_size=187,
    ),
    experiment_name="1D CNN Baseline",
    trainstreamer=trainstreamer,
    validstreamer=validstreamer,
    teststreamer=teststreamer,
    loss_fn=nn.CrossEntropyLoss(),
    class_weights=False,
    parallel=False,
    runs=3,
    epochs=5,
    filename="results/results.csv",
)

In [ ]:
# ============================================
# Experiment 2 - 1D CNN met class weights
# ============================================

In [ ]:
# Class weights uitsluitend uit de trainingssubset berekenen
class_weights = calculate_class_weights(traindataset)
classes = ["N", "S", "V", "F", "Q"]
for cls, weight in zip(classes, class_weights):
    print(f"{cls}: weight={weight:.3f}")


In [ ]:
run_experiment(
    model_function=lambda: CNN1D(
        num_classes=5,
        filters=16,
        input_size=187,
    ),
    experiment_name="1D CNN Weighted",
    trainstreamer=trainstreamer,
    validstreamer=validstreamer,
    teststreamer=teststreamer,
    loss_fn=nn.CrossEntropyLoss(
        weight=class_weights,
    ),
    class_weights=True,
    parallel=False,
    runs=3,
    epochs=5,
    filename="results/results.csv",
)